# 08 · Custom training loop

`NextSequence` is a thin layer over public building blocks. Use them directly when you need
something it doesn't do (a different optimizer, logging, early stopping, Lightning, ...).

| Building block | What it is |
|---|---|
| `NextSequenceModel` | a plain `nn.Module`, `forward(X, mask)` → raw outputs |
| `calculate_loss` | per-feature losses, chosen by each feature's task type |
| `UncertaintyWeightedLoss` | combines them with learned task weights |
| `next_sequence.train` | `build_optimizer`, `build_scheduler`, `train_one_epoch`, `evaluate`, `to_device` |

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

import math
import torch
from torch.utils.data import DataLoader
from next_sequence import (CoreDataset, CustomCollate, Feature, NextSequence,
                           NextSequenceConfig, TaskType, create_bow, make_collate)

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {"item": make_collate("long"), "price": make_collate("float"), "weekday": make_collate("long")}
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)

def loader(samples, inference=False, shuffle=False, batch_size=32):
    collate = CustomCollate.from_config(config, "item", funcs, inference=inference)
    return DataLoader(CoreDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

print(f"{len(users)} users | train/val/test samples: {len(train)}/{len(val)}/{len(test)} | vocab: {len(bow)}")

300 users | train/val/test samples: 300/300/300 | vocab: 7


## Level 1: the ready-made epoch functions

In [2]:
from next_sequence import NextSequenceModel, UncertaintyWeightedLoss, default_device
from next_sequence.train import build_optimizer, build_scheduler, evaluate, train_one_epoch

device = default_device()
torch.manual_seed(0)
net = NextSequenceModel(config).to(device)
criterion = UncertaintyWeightedLoss(features).to(device)       # holds the learned task weights

train_loader, val_loader = loader(train, shuffle=True), loader(val)
epochs = 10
optimizer = build_optimizer(net, criterion, lr=3e-3)          # includes criterion.parameters()
scheduler = build_scheduler(optimizer, total_steps=epochs * len(train_loader))

for epoch in range(1, epochs + 1):
    tr = train_one_epoch(net, criterion, train_loader, optimizer, scheduler, device)
    va = evaluate(net, criterion, val_loader, device)
    print(f"epoch {epoch:2d} | train item {tr['item']:.4f} price {tr['price']:.4f} "
          f"| val item {va['item']:.4f} price {va['price']:.4f}")

epoch  1 | train item 1.7423 price 1.1727 | val item 1.4878 price 0.5541
epoch  2 | train item 1.1870 price 0.4474 | val item 1.0059 price 0.3939


epoch  3 | train item 0.7247 price 0.3262 | val item 0.5166 price 0.2035
epoch  4 | train item 0.3710 price 0.1053 | val item 0.2377 price 0.1158


epoch  5 | train item 0.1642 price 0.0675 | val item 0.0992 price 0.0505
epoch  6 | train item 0.0752 price 0.0430 | val item 0.0534 price 0.0342


epoch  7 | train item 0.0515 price 0.0344 | val item 0.0479 price 0.0285
epoch  8 | train item 0.0408 price 0.0331 | val item 0.0320 price 0.0234


epoch  9 | train item 0.0336 price 0.0303 | val item 0.0304 price 0.0220
epoch 10 | train item 0.0330 price 0.0280 | val item 0.0303 price 0.0215


## Level 2: write the loop yourself

This is what `train_one_epoch` does. Two details matter:

- the optimizer must include `criterion.parameters()`, or the task weights never learn;
- don't apply weight decay to `criterion.log_vars` (`build_optimizer` handles both).

Here with plain `torch.optim.Adam` and simple early stopping:

In [3]:
from next_sequence import calculate_loss
from next_sequence.train import to_device

torch.manual_seed(0)
net = NextSequenceModel(config).to(device)
criterion = UncertaintyWeightedLoss(features).to(device)
optimizer = torch.optim.Adam(list(net.parameters()) + list(criterion.parameters()), lr=3e-3)

best_val, patience, bad_epochs = float("inf"), 3, 0
for epoch in range(1, 31):
    net.train()
    for batch in train_loader:
        batch = to_device(batch, device)
        outputs = net(batch["X"], batch["mask"])
        losses = calculate_loss(outputs, batch["Y"], features)   # {"item": ..., "price": ...}
        loss = criterion(losses)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    val_metrics = evaluate(net, criterion, val_loader, device)
    improved = val_metrics["item"] < best_val - 1e-4
    best_val, bad_epochs = (val_metrics["item"], 0) if improved else (best_val, bad_epochs + 1)
    print(f"epoch {epoch:2d} | val item loss {val_metrics['item']:.4f}{'  *' if improved else ''}")
    if bad_epochs >= patience:
        print(f"stopping: no improvement for {patience} epochs")
        break

epoch  1 | val item loss 1.5108  *
epoch  2 | val item loss 1.0547  *
epoch  3 | val item loss 0.6046  *


epoch  4 | val item loss 0.2269  *
epoch  5 | val item loss 0.1168  *
epoch  6 | val item loss 0.0580  *


epoch  7 | val item loss 0.0360  *
epoch  8 | val item loss 0.0252  *
epoch  9 | val item loss 0.0198  *


epoch 10 | val item loss 0.0157  *
epoch 11 | val item loss 0.0129  *
epoch 12 | val item loss 0.0110  *


epoch 13 | val item loss 0.0094  *
epoch 14 | val item loss 0.0082  *
epoch 15 | val item loss 0.0073  *


epoch 16 | val item loss 0.0064  *
epoch 17 | val item loss 0.0057  *


epoch 18 | val item loss 0.0051  *
epoch 19 | val item loss 0.0047  *
epoch 20 | val item loss 0.0043  *


epoch 21 | val item loss 0.0040  *
epoch 22 | val item loss 0.0035  *
epoch 23 | val item loss 0.0032  *


epoch 24 | val item loss 0.0029  *
epoch 25 | val item loss 0.0026  *
epoch 26 | val item loss 0.0024  *


epoch 27 | val item loss 0.0022  *
epoch 28 | val item loss 0.0021  *


epoch 29 | val item loss 0.0020
epoch 30 | val item loss 0.0018  *


The learned task weights:

In [4]:
criterion.weights()

{'item': 1.803325891494751, 'price': 1.2432222366333008}

## Predicting with the bare model

Call `eval()`, disable gradients, and read the outputs. This is what `NextSequence.predict` wraps.

In [5]:
net.eval()
with torch.no_grad():
    batch = to_device(next(iter(loader(test))), device)
    outputs = net(batch["X"], batch["mask"])
id2item = {i: t for t, i in bow.items()}
print("predicted:", [id2item[i] for i in outputs["item"].argmax(-1)[:8].tolist()])
print("true     :", [u["item"][-1] for u in test_raw[:8]])

predicted: ['eggs', 'bread', 'dates', 'cheese', 'dates', 'eggs', 'cheese', 'eggs']
true     : ['eggs', 'bread', 'dates', 'cheese', 'dates', 'eggs', 'cheese', 'eggs']
